In [ ]:
import pvlib
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import os
from plotly.subplots import make_subplots
import satlib


In [ ]:
### Location
lat = 55.6 #degrees North
lon = 12.1 #degrees East
albedo = 0.2

### Dates
start = pd.Timestamp('2023-01-01 00:00:00', tz='Europe/Copenhagen')
end = pd.Timestamp('2023-12-31 23:59:00', tz='Europe/Copenhagen')

### Module parameters
module_width = 1
module_length = 1.6

### System parameters
pitch = 11.75 #estimated from other measurements
length = 3.3 #estimated from other measurements
hub_height = 1.95
gcr = 0.28
max_angle = 60
axis_tilt = 0
axis_azimuth = 180

skymodel = 'perez-driesse' #options: perez, perez-driesse, haydavies, isotropic

In [ ]:
# create location variable
location = pvlib.location.Location(latitude=lat, longitude=lon)
if os.path.isfile(f"data/cams/cams_radiation_{lat}_{lon}_{start}_{end}.csv"):
    data = pd.read_csv(f'data/cams/cams_radiation_{lat}_{lon}_{start}_{end}.csv', index_col=0, parse_dates=True)
else:
    data, meta = pvlib.iotools.get_cams(lat, lon,start,end,'arule@dtu.dk', identifier='cams_radiation',time_step='1min') 
    data.to_csv(f'data/cams/cams_radiation_{lat}_{lon}_{start}_{end}.csv')

In [ ]:
# get solar position and module tilt angle
solar_position = location.get_solarposition(data.index)

#get reference module tilt and POA
module_tilt_ref = pvlib.tracking.singleaxis(solar_position['zenith'], solar_position['azimuth'],backtrack=True, gcr=gcr, max_angle=max_angle,axis_azimuth=180)
surface_orientation_ref = pvlib.tracking.calc_surface_orientation(module_tilt_ref['tracker_theta'], axis_tilt = axis_tilt, axis_azimuth=axis_azimuth)

dni_extra = pvlib.irradiance.get_extra_radiation(solar_position['zenith'])

model_output_ref = pvlib.bifacial.ants2d.get_irradiance(
    tracker_rotation=module_tilt_ref['tracker_theta'], 
    axis_azimuth=180, 
    solar_zenith=solar_position['zenith'], 
    solar_azimuth=solar_position['azimuth'], 
    gcr=gcr, 
    height=hub_height, 
    pitch=pitch, 
    ghi=data['ghi'], dhi=data['dhi'], dni=data['dni'],
    albedo=albedo,
    model=skymodel,)

poa_total_ref = np.asarray(model_output_ref['poa_front'])


In [ ]:
# CENER "diffuse radiation algorithm" (Munoz 2024)
# If GHI > GPOA, tilt = 0, otherwise tilt = reference tilt

# Find timestamps where GHI > GPOA
flat_tracking_flag = data['ghi'] > poa_total_ref

# take the reference tracking angle if GHI < GPOA, and 0 if GHI > GPOA
tracker_angle_set = np.where(flat_tracking_flag, 0, module_tilt_ref['tracker_theta'])

#re-run the model with these tracker angles. 
# This is not the fastest or most efficient approach 
# (could re-use results where tracker angle is the same)

model_output_test = pvlib.bifacial.ants2d.get_irradiance(
    tracker_rotation=tracker_angle_set, 
    axis_azimuth=180, 
    solar_zenith=solar_position['zenith'], 
    solar_azimuth=solar_position['azimuth'], 
    gcr=gcr, 
    height=hub_height, 
    pitch=pitch, 
    ghi=data['ghi'], dhi=data['dhi'], dni=data['dni'],
    albedo=albedo,
    model=skymodel,)

poa_total_test = np.asarray(model_output_test['poa_front'])



In [ ]:
poa_gain_instantaneous = 100*(poa_total_test - poa_total_ref) / poa_total_ref

poa_gain_total = 100*(np.nansum(poa_total_test) - np.nansum(poa_total_ref)) / np.nansum(poa_total_ref)

poa_gain_total

In [ ]:
# CENER "analytical model"
# eq. 12 in I. Muñoz, A. Guinda, G. Olivares, S. Díaz, A. M. Gracia-Amillo, and L. Casajús, 
# “Evaluation of Horizontal Single-Axis Solar Tracker Algorithms in Terms of Energy Production 
# and Operational Performance,” Solar RRL, vol. 8, no. 1, p. 2300507, 2024, doi: 10.1002/solr.202300507.

# troubleshooting Qs:   1. what were the units of beta_opt in the paper....?
#                       2. is the azimuth difference right? 
#                           look at solar_position['azimuth']-surface_orientation_ref['surface_azimuth']
#                           yes, the azimuth difference is correct, looks good in plotting

zenith = np.deg2rad(np.asarray(solar_position["zenith"], dtype=float))
solar_azimuth = np.deg2rad(np.asarray(solar_position["azimuth"], dtype=float))
surface_azimuth = np.deg2rad(np.asarray(surface_orientation_ref["surface_azimuth"], dtype=float))

azimuth_difference = solar_azimuth - surface_azimuth

dni = np.asarray(data["dni"], dtype=float)
dhi = np.asarray(data["dhi"], dtype=float)
ghi = np.asarray(data["ghi"], dtype=float)

numerator = (dni* np.sin(zenith)* np.cos(azimuth_difference))
denominator = ((dhi - ghi * albedo) / 2+ dni * np.cos(zenith))

beta_optimal = np.rad2deg(np.arctan2(numerator, denominator))

# beta_optimal = np.rad2deg(np.arctan((dni)*np.sin(np.radians(solar_position['zenith']))*\
#                np.cos(np.radians((solar_position['azimuth'])-surface_orientation_ref['surface_azimuth'])))/ \
#                ((dhi-ghi*albedo)/2)+dni*np.cos(np.radians((solar_position['zenith'])))))

beta_optimal = np.clip(beta_optimal, -max_angle, max_angle) #clip to max angle

beta_optimal

In [ ]:
beta_angle_analytical = satlib.diffuse_optimization.munoz2024_analytical(solar_position["zenith"], solar_position["azimuth"], \
                                                   data["dni"], data["dhi"], data["ghi"], albedo)
print(f"beta_angle_analytical: {beta_angle_analytical}")
beta_angle_analytical_2 = np.clip(beta_angle_analytical, -max_angle, max_angle) #clip to max angle
print(f"beta_angle_analytical after clipping: {beta_angle_analytical_2}")

beta_angle_diffuse = satlib.diffuse_optimization.munoz2024_diffuse(module_tilt_ref['tracker_theta'],solar_position["zenith"], solar_position["azimuth"], \
                                                   dni_extra,data["dni"], data["dhi"], data["ghi"], albedo)
beta_angle_diffuse = np.clip(beta_angle_diffuse, -max_angle, max_angle) #clip to max angle


In [ ]:

fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(
    go.Scatter(
        x=data.index,
        y=np.asarray(beta_angle_analytical),
        mode="lines",
        name="beta_angle_analytical",
        line=dict(color="red"),
    ),
)

fig.add_trace(
    go.Scatter(
        x=data.index,
        y=np.asarray(beta_angle_analytical_2),
        mode="lines",
        name="beta_angle_analytical_2",
        line=dict(color="blue"),
    ),
)

In [ ]:
fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(
    go.Scatter(
        x=data.index,
        y=np.asarray(module_tilt_ref['surface_tilt']),
        mode="lines",
        name="ref tilt",
        line=dict(color="red"),
    ),
)

fig.add_trace(
    go.Scatter(
        x=data.index,
        y=np.asarray(beta_optimal),
        mode="lines",
        name="optimal tilt",
        line=dict(color="blue"),
    ),
)

fig.add_trace(
    go.Scatter(
        x=data.index,
        y=np.asarray(beta_angle_analytical),
        mode="lines",
        name="munoz analytical",
        line=dict(color="green"),
    ),
)

fig.add_trace(
    go.Scatter(
        x=data.index,
        y=np.asarray(beta_angle_diffuse),
        mode="lines",
        name="munoz diffuse",
        line=dict(color="orange"),
    ),
)




# fig.add_trace(
#     go.Scatter(
#         x=data.index,
#         y=np.asarray(solar_position['azimuth']-surface_orientation_ref['surface_azimuth']),
#         mode="lines",
#         name="delta azimuth",
#         line=dict(color="blue"),
#     ),
# )

# fig.add_trace(
#     go.Scatter(
#         x=data.index,
#         y=np.asarray(solar_position['azimuth']),
#         mode="lines",
#         name="solpos azimuth",
#         line=dict(color="green"),
#     ),
# )

# fig.add_trace(
#     go.Scatter(
#         x=data.index,
#         y=np.asarray(surface_orientation_ref['surface_azimuth']),
#         mode="lines",
#         name="surface azimuth",
#         line=dict(color="purple"),
#     ),
# )

In [ ]:
fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(
    go.Scatter(
        x=data.index,
        y=np.asarray(poa_total_ref),
        mode="lines",
        name="POA ref",
        line=dict(color="red"),
    ),
)

fig.add_trace(
    go.Scatter(
        x=data.index,
        y=np.asarray(poa_total_test),
        mode="lines",
        name="POA optimal",
        line=dict(color="blue"),
    ),
)

fig.add_trace(
    go.Scatter(
        x=data.index,
        y=np.asarray(poa_gain_instantaneous),
        mode="lines",
        name="POA gain instantaneous",
        line=dict(color="green"),
        
    ),
    secondary_y=True,
)